# Standardized multi-feature score

In [ ]:
import numpy as np
import pandas as pd

def synthetic_prices():
    rng = np.random.default_rng(7)
    returns = rng.normal(0.0005, 0.012, size=(120, 4))
    return pd.DataFrame(100.0 * (1.0 + returns).cumprod(axis=0), columns=list("ABCD"))


In [ ]:
from sklearn.preprocessing import StandardScaler

def signal(prices):
    """Standardize lagged features with sklearn, then average them into a score."""
    rets = prices.pct_change().shift(1)
    feats = pd.concat({"r5": rets.rolling(5).mean(), "v20": rets.rolling(20).std()}, axis=1).fillna(0.0)
    mu = feats.expanding(min_periods=20).mean()
    sd = feats.expanding(min_periods=20).std().replace(0.0, np.nan)
    scaled = ((feats - mu) / sd).fillna(0.0)
    return (scaled["r5"] - scaled["v20"]).clip(-2, 2)


In [ ]:
def backtest(prices, cost_bps=10):
    positions = signal(prices)
    asset_returns = prices.pct_change().fillna(0.0)
    gross = (positions * asset_returns).mean(axis=1)
    turnover = positions.diff().abs().sum(axis=1).fillna(0.0)
    return gross - turnover * cost_bps / 10_000

def sharpe(pnl):
    pnl_std = pnl.std()
    return float(pnl.mean() / pnl_std * np.sqrt(252)) if pnl_std > 0 else 0.0

print(round(sharpe(backtest(synthetic_prices())), 3))
